In [1]:
import pandas as pd

# 1. 读取原始 Procedures 数据
raw = pd.read_excel(
    "surgery patient1-5.xlsx",
    sheet_name="Raw_Procedures",
    dtype={"Patient_No": "string"}
)

# 2. 读取组员的患者匿名化对照表
registry = pd.read_csv(
    "person_registry.csv",
    dtype={"Patient_No": "string"}
)

# 3. 根据 Patient_No 匹配患者
data = raw.merge(
    registry[["Patient_No", "person_id", "date_offset_days"]],
    on="Patient_No",
    how="left",
    validate="many_to_one"
)

# 4. 检查有没有匹配失败
if data["date_offset_days"].isna().any():
    raise ValueError("Some patients have no date offset!")

# 5. 计算新的手术日期
data["operation_date"] = pd.to_datetime(data["operation_date"])

data["procedure_date"] = (
    data["operation_date"]
    + pd.to_timedelta(data["date_offset_days"], unit="D")
).dt.date

# 6. 查看结果
print(data[["person_id", "procedure_date"]])

    person_id procedure_date
0  2018449483     2016-06-13
1  2018449483     2016-03-25
2  1702676777     2016-06-23
3  1702676777     2016-06-23
4   245249064     2016-08-23
5   245249064     2016-05-03
6  1507501565     2016-03-29
7  1575455504     2016-08-27
